# 03 - Experience B : interference des artefacts (SRM)

Hypothese H2 : sur les images generees, la trace de l'insertion est masquee par les artefacts de
generation. On mesure l'ampleur du deplacement cover vers stego par une distance multivariee, sans
detecteur entraine. Avec SRM, la mesure vaut aussi pour les algorithmes adaptatifs, contrairement a SPAM.

Figures nommees selon expB_srm_<type>_<algo>_p<charge>.png.

## Configuration

In [ ]:
# ================= CONFIGURATION =================
FEATURE  = 'srm'
ALGOS    = ['lsb', 'uniward', 'hill']
PAYLOAD  = 0.4
SOURCES  = ['natural', 'sd', 'sdxl', 'adm']
SEED     = 42
# ================================================
pp = str(PAYLOAD).replace('.', '')
print('Experience B sur', FEATURE, '| algos', ALGOS, '| charge', PAYLOAD)

In [ ]:
import os, glob
import numpy as np
np.random.seed(SEED)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/memoire_data'
except Exception:
    DATA_DIR = os.path.abspath('./memoire_data')
FEAT_DIR = f'{DATA_DIR}/features_{FEATURE}'
FIG = f'{DATA_DIR}/figures'
os.makedirs(FIG, exist_ok=True)
print('Caracteristiques :', FEAT_DIR, '| Figures :', FIG)

In [ ]:
!pip install -q scikit-learn matplotlib
print('Installation terminee.')

## 1. Distance multivariee cover vers stego

Pour chaque source et chaque algorithme, la distance de Mahalanobis entre les distributions cover et
stego, dans l'espace complet des caracteristiques, avec covariance regularisee. Une distance plus faible
sur une source generee que sur le naturel signale l'interference.

In [ ]:
from sklearn.covariance import LedoitWolf

def charger(source, setname):
    return np.load(f'{FEAT_DIR}/{source}__{setname}.npy')

def distance(source, algo):
    Xc = charger(source, 'cover')
    Xs = charger(source, f'{algo}_p{PAYLOAD}')
    n = min(len(Xc), len(Xs)); Xc, Xs = Xc[:n], Xs[:n]
    both = np.vstack([Xc, Xs]); mu, sd = both.mean(0), both.std(0) + 1e-9
    Xc, Xs = (Xc - mu) / sd, (Xs - mu) / sd
    cov = LedoitWolf().fit(np.vstack([Xc, Xs])).covariance_
    diff = Xs.mean(0) - Xc.mean(0)
    return float(np.sqrt(diff @ np.linalg.pinv(cov) @ diff))

D = {a: {s: distance(s, a) for s in SOURCES} for a in ALGOS}
print(f"{'algo':8s} " + ' '.join(f'{s:>9s}' for s in SOURCES))
for a in ALGOS:
    print(f'{a:8s} ' + ' '.join(f'{D[a][s]:9.3f}' for s in SOURCES))

## 2. Figure : distances par source, un graphe par algorithme

In [ ]:
import matplotlib.pyplot as plt
palette = {'natural': 'tab:blue', 'sd': 'tab:orange', 'sdxl': 'tab:green', 'adm': 'tab:red'}

for a in ALGOS:
    fig, ax = plt.subplots(figsize=(7, 5))
    vals = [D[a][s] for s in SOURCES]
    ax.bar(SOURCES, vals, color=[palette[s] for s in SOURCES])
    if 'natural' in D[a]:
        ax.axhline(D[a]['natural'], ls='--', color='gray', label='niveau naturel')
    ax.set_ylabel('distance cover vers stego'); ax.legend()
    ax.set_title(f'Interference, {a} {PAYLOAD} bpp, SRM')
    nom = f'{FIG}/expB_srm_maha_{a}_p{pp}.png'
    plt.tight_layout(); plt.savefig(nom, dpi=150); plt.show()
    print('figure :', os.path.basename(nom))

## 3. Figure : projection 2D des classes, un graphe par algorithme

Projection des caracteristiques SRM en deux dimensions, pour voir le chevauchement des paires cover et
stego selon la source. Un chevauchement plus fort sur une source generee illustre l'interference.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

M = 300   # on limite le nombre d'images par classe pour une figure lisible
for a in ALGOS:
    X, couleurs, noms = [], [], []
    for s in SOURCES:
        Xc = charger(s, 'cover')[:M]; Xs = charger(s, f'{a}_p{PAYLOAD}')[:M]
        for classe, Dset in [('cover', Xc), ('stego', Xs)]:
            X.append(Dset); couleurs += [palette[s]] * len(Dset); noms += [f'{s} {classe}'] * len(Dset)
    X = np.vstack(X)
    P = PCA(n_components=2, random_state=SEED).fit_transform(StandardScaler().fit_transform(X))
    fig, ax = plt.subplots(figsize=(8, 7)); vus = set()
    for i in range(len(P)):
        marqueur = 'o' if 'cover' in noms[i] else '^'
        lab = noms[i] if noms[i] not in vus else None; vus.add(noms[i])
        ax.scatter(P[i, 0], P[i, 1], c=couleurs[i], marker=marqueur, s=10, alpha=0.5, label=lab)
    ax.set_title(f'Projection 2D SRM, {a} {PAYLOAD} bpp, rond cover, triangle stego')
    ax.legend(fontsize=8)
    nom = f'{FIG}/expB_srm_proj2d_{a}_p{pp}.png'
    plt.tight_layout(); plt.savefig(nom, dpi=150); plt.show()
    print('figure :', os.path.basename(nom))

## Suite

On copie les figures dans le dossier results du depot et on renseigne leur interpretation dans
results/figures_interpretations.md, chaque figure etant referencee par son nom.